# 🧬 Práctica C: Ensamblaje de Genomas con Python + conda en Google Colab



> [!IMPORTANT]
> **Antes de continuar**, lea la [guía de prácticas compartida](00_genome_assembly_common.md) — contiene el contexto biológico, los tres casos de estudio y las instrucciones para descargar los datos en Colab.

Esta práctica sigue el mismo **flujo de trabajo** que las Prácticas A y B, pero en lugar de Galaxy usa **Google Colab** como plataforma de cómputo y **Python** y **Conda** para instalar las herramientas bioinformáticas necesarias. Esto le permite:

- Ejecutar herramientas de línea de comandos (`fastp`, `SPAdes`, `QUAST`) desde un notebook en la nube.
- Analizar y visualizar los resultados con Python (`pandas`, `matplotlib`).
- Entender cómo se encadenan estas herramientas en un pipeline real.

## ⚙️ Paso 0 — Configurar el entorno con conda

Ejecute esta celda **primero**. Instala Miniconda y todas las herramientas necesarias.
⏱️ Puede tardar 8–12 minutos.

In [ ]:
# Instalamos biopython y Conda para colab
!pip install condacolab biopython --quiet

import condacolab
condacolab.install() # Esto reinicia el kernel automáticamente

# Verify condacolab installation
import os
os.environ['PATH'] = '/usr/local/bin:/opt/conda/bin:' + os.environ['PATH']

!conda --version

Una vez que Conda esté instalado, es posible que necesites reiniciar el entorno de ejecución de Colab (Runtime > Restart runtime) para que los cambios surtan efecto. Después de reiniciar, puedes ejecutar comandos `conda` en nuevas celdas de código.

### Instalar herramientas

Ya que hemos instalado **Conda** es necesario instalar las herramientas que vamos a usar para el ensamblaje.

In [ ]:
# Instalar herramientas bioinformáticas

!conda install -y \
  -c bioconda \
  -c conda-forge \
  fastqc \
  trimmomatic \
  bwa \
  samtools \
  bcftools \
  fastp \
  spades \
  quast
!echo "✅ Herramientas bioinformáticas instaladas"

ahora vamos a instalar otras herramientas propias de **Python** que nos ayudaran a procesar y ver los resutlados mas facil para los humanos

In [ ]:
!pip install matplotlib pandas --quiet

# Imports centralizados — todos en un solo lugar para evitar duplicación
import json
import os
import pathlib
import glob
import subprocess

import matplotlib.pyplot as plt
import pandas as pd

os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

print("✅ Dependencias Python listas")

## 📥 Paso 1 — Descargar los datos del caso asignado

Solo trabajaremos con un solo caso, por lo cual debe definir primero el caso que le indicó el profesor (A, B, C o D). Asigne a la variable el caso que va a trabajar.

Consulte la [guía compartida](00_genome_assembly_common.md#-casos-de-estudio) para el contexto biológico de cada caso.

In [ ]:
# ⚠️ **IMPORTANTE**: Define el caso a trabajar aquí y solo aquí.
# Descomenta la línea que corresponda a tu caso (A, B, C o D).
CASO = "caso_A" # Por ejemplo, para caso A

In [ ]:
# Crear directorio de datos
os.makedirs(f"{CASO}/data", exist_ok=True)

# Diccionario de URLs por caso
CASOS_CONFIG = {
    "caso_A": {
        "descripcion": "Staphylococcus aureus MRSA (~2.8 Mb, GC ~33%)",
        "r1": "ftp://ftp.sra.ebi.ac.uk/vol1/fastq/DRR187/DRR187559/DRR187559_1.fastq.gz",
        "r2": "ftp://ftp.sra.ebi.ac.uk/vol1/fastq/DRR187/DRR187559/DRR187559_2.fastq.gz",
        "ref": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/013/425/GCF_000013425.1_ASM1342v1/GCF_000013425.1_ASM1342v1_genomic.fna.gz"
    },
    "caso_B": {
        "descripcion": "Klebsiella pneumoniae (~5.5 Mb, GC ~57%)",
        "r1": "ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR148/071/ERR14828471/ERR14828471_1.fastq.gz",
        "r2": "ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR148/071/ERR14828471/ERR14828471_2.fastq.gz",
        "ref": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/240/185/GCF_000240185.1_ASM24018v2/GCF_000240185.1_ASM24018v2_genomic.fna.gz"
    },
    "caso_C": {
        "descripcion": "Streptomyces venezuelae (~8.2 Mb, GC ~72%)",
        "r1": "ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/006/SRR2589046/SRR2589046_1.fastq.gz",
        "r2": "ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/006/SRR2589046/SRR2589046_2.fastq.gz",
        "ref": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/253/235/GCF_000253235.1_ASM25323v1/GCF_000253235.1_ASM25323v1_genomic.fna.gz"
    },
    "caso_D": {
        "descripcion": "Pseudomonas abieticivorans (~6.7 Mb, GC ~63%)",
        "r1": "ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR246/000/SRR24684300/SRR24684300_1.fastq.gz",
        "r2": "ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR246/000/SRR24684300/SRR24684300_2.fastq.gz",
        "ref": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/023/509/015/GCF_023509015.1_ASM2350901v1/GCF_023509015.1_ASM2350901v1_genomic.fna.gz"
    }
}

if CASO not in CASOS_CONFIG:
    raise ValueError(f"⚠️ CASO '{CASO}' no reconocido. Use 'caso_A', 'caso_B', 'caso_C' o 'caso_D'.")

cfg = CASOS_CONFIG[CASO]
print(f"── {CASO.upper()}: {cfg['descripcion']} ──")

# Descargar archivos
r1_path  = f"{CASO}/data/reads_1.fastq.gz"
r2_path  = f"{CASO}/data/reads_2.fastq.gz"
ref_gz   = f"{CASO}/data/ref.fna.gz"
ref_path = f"{CASO}/data/ref.fna"

print("Descargando R1...")
!wget -q "{cfg['r1']}" -O "{r1_path}"
print("Descargando R2...")
!wget -q "{cfg['r2']}" -O "{r2_path}"
print("Descargando referencia...")
!wget -q "{cfg['ref']}" -O "{ref_gz}"

# FIX: verificar que los archivos existen y tienen tamaño > 0
for fpath in [r1_path, r2_path, ref_gz]:
    size = os.path.getsize(fpath) if os.path.exists(fpath) else 0
    status = "✅" if size > 1000 else "❌ VACÍO o faltante"
    print(f"{status}  {fpath}  ({size:,} bytes)")

# FIX: se corrigió la variable gz_file_path que antes era indefinida
print(f"\nDescomprimiendo {ref_gz}...")
!gunzip -f "{ref_gz}"
print(f"✅ Caso {CASO} listo")

## 👀 Paso 1.1 — Inspeccionar los archivos descargados

Vamos a verificar que los archivos FASTQ y el genoma de referencia FASTA se hayan descargado correctamente y tengan el formato esperado. Revisaremos el encabezado de cada uno.

In [ ]:
print(f"=== Encabezado del archivo FASTQ R1 para {CASO} ===")
!zcat "{r1_path}" | head -n 8

print(f"\n=== Encabezado del archivo FASTQ R2 para {CASO} ===")
!zcat "{r2_path}" | head -n 8

print(f"\n=== Encabezado del archivo FASTA de referencia para {CASO} ===")
!head -n 5 "{ref_path}"

## 🔎 Paso 2 — Control de calidad y limpieza con fastp

`fastp` realiza en un solo paso: detección y recorte de adaptadores, filtrado por calidad y generación de reporte HTML + JSON.

Parámetros clave:
- `--qualified_quality_phred 30`: descarta bases con Phred < Q30
- `--length_required 50`: descarta lecturas más cortas de 50 pb tras el trimming

In [ ]:
os.makedirs(f"{CASO}/results/qc_results", exist_ok=True)

clean_r1 = f"{CASO}/results/clean_R1.fastq.gz"
clean_r2 = f"{CASO}/results/clean_R2.fastq.gz"
fastp_html = f"{CASO}/results/qc_results/fastp_report.html"
fastp_json = f"{CASO}/results/qc_results/fastp_report.json"

# rutas explícitas en lugar de wildcards (*_1.fastq.gz) que fastp no expande
!fastp \
  --in1  "{r1_path}" \
  --in2  "{r2_path}" \
  --out1 "{clean_r1}" \
  --out2 "{clean_r2}" \
  --html "{fastp_html}" \
  --json "{fastp_json}" \
  --qualified_quality_phred 30 \
  --length_required 50 \
  --thread 4

print(f"✅ Limpieza completa. Reporte en {fastp_html}")

## 📊 Paso 3 — Explorar el reporte de fastp con Python

Leemos el JSON generado por fastp y mostramos un resumen comparativo antes/después del filtrado.

In [ ]:
report = json.loads(pathlib.Path(fastp_json).read_text())

summary = report["summary"]
before  = summary["before_filtering"]
after   = summary["after_filtering"]

print("=== Resumen de calidad ===")
print(f"{'':30s} {'Antes':>12} {'Después':>12}")
print("-" * 56)
print(f"{'Lecturas totales':30s} {before['total_reads']:>12,} {after['total_reads']:>12,}")
print(f"{'Bases totales':30s} {before['total_bases']:>12,} {after['total_bases']:>12,}")
print(f"{'Calidad Q20 (%)':30s} {before['q20_rate']*100:>11.1f}% {after['q20_rate']*100:>11.1f}%")
print(f"{'Calidad Q30 (%)':30s} {before['q30_rate']*100:>11.1f}% {after['q30_rate']*100:>11.1f}%")
print(f"{'GC (%)':30s} {before['gc_content']*100:>11.1f}% {after['gc_content']*100:>11.1f}%")

### Comparación de Métricas de Calidad (Antes vs. Después de `fastp`)

Graficamos el Phred score medio por posición en Read 1 y Read 2. Las zonas de color indican:
- 🔴 Rojo: Q < 20 (calidad baja, no confiable)
- 🟡 Amarillo: 20 ≤ Q < 30 (calidad media)
- 🟢 Verde: Q ≥ 30 (calidad alta, recomendada)

In [ ]:
# se usa la curva 'mean' directamente del JSON de fastp
def get_mean_quality_curve(read_section):
    """Extrae la curva de calidad media por posición del reporte fastp."""
    curves = read_section.get('quality_curves', {})
    mean_q = curves.get('mean', [])
    if not mean_q:
        return [], []
    positions = list(range(1, len(mean_q) + 1))
    return positions, mean_q

pos_r1_before, q_r1_before = get_mean_quality_curve(report['read1_before_filtering'])
pos_r1_after,  q_r1_after  = get_mean_quality_curve(report['read1_after_filtering'])
pos_r2_before, q_r2_before = get_mean_quality_curve(report['read2_before_filtering'])
pos_r2_after,  q_r2_after  = get_mean_quality_curve(report['read2_after_filtering'])

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
fig.suptitle(f'Calidad Media por Base — {CASO}', fontsize=14, fontweight='bold')

for ax, title, pos_b, q_b, pos_a, q_a in [
    (axes[0], 'Read 1', pos_r1_before, q_r1_before, pos_r1_after, q_r1_after),
    (axes[1], 'Read 2', pos_r2_before, q_r2_before, pos_r2_after, q_r2_after),
]:
    ax.axhspan(0,  20, color='red',    alpha=0.1, label='Calidad Baja (Q<20)')
    ax.axhspan(20, 30, color='yellow', alpha=0.1, label='Calidad Media (20≤Q<30)')
    ax.axhspan(30, 40, color='green',  alpha=0.1, label='Calidad Alta (Q≥30)')

    if pos_b and q_b:
        ax.plot(pos_b, q_b, color='red',   alpha=0.8, label='Antes de fastp')
    if pos_a and q_a:
        ax.plot(pos_a, q_a, color='green', alpha=0.8, label='Después de fastp')

    ax.set_title(title)
    ax.set_xlabel('Posición de Base (Ciclo)')
    ax.set_ylabel('Calidad Media (Phred Score)')
    ax.set_ylim(0, 40)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.legend()

plt.tight_layout()

# FIX: savefig ANTES de show (show() vacía la figura)
output_dir = f"{CASO}/results/qc_results"
plot_path  = os.path.join(output_dir, f"quality_plot_{CASO}.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Gráfico guardado en {plot_path}")

## 🔧 Paso 4 — Ensamblaje de novo con SPAdes

SPAdes (St. Petersburg Assembler) es un software de ensamblaje de novo que se utiliza para reconstruir genomas a partir de lecturas de secuencias de ADN. Está diseñado para trabajar con lecturas de secuenciación de alto rendimiento (como Illumina) y puede manejar diferentes tipos de datos, incluyendo lecturas emparejadas (paired-end) y lecturas de longitud variable. SPAdes es conocido por su precisión y capacidad para ensamblar genomas complejos, y es una herramienta ampliamente utilizada en bioinformática para proyectos de secuenciación de genomas bacterianos y eucariotas pequeños.

NOTA: Este paso demora entre 30-50 minutos

In [ ]:
spades_out = f"{CASO}/results/spades_assembly"

# Se usa --isolate para muestras bacterianas puras (recomendado)
!spades.py \
  -1 "{clean_r1}" \
  -2 "{clean_r2}" \
  -o "{spades_out}" \
  --isolate \
  -t 4 \
  -m 12

contigs_path = f"{spades_out}/contigs.fasta"
if os.path.exists(contigs_path):
    print(f"✅ Ensamblaje completo. Contigs en {contigs_path}")
else:
    print("❌ No se encontró contigs.fasta — revise los logs de SPAdes arriba.")

## 📏 Paso 5 — Evaluación del ensamblaje con QUAST

QUAST compara los contigs obtenidos contra el genoma de referencia para calcular métricas de calidad.
Ver la definición de N50, L50 y Genome Fraction en las secciones **6** y **7.5** del [README del Módulo 5](../README.md).

In [ ]:
quast_out = f"{CASO}/results/quast_results"

!quast.py \
  "{contigs_path}" \
  -r "{ref_path}" \
  -o "{quast_out}" \
  --threads 4

print(f"✅ QUAST completo. Reporte en {quast_out}/report.html")

## 📊 Paso 6 — Leer y visualizar los resultados de QUAST con Python

Leemos el reporte TSV de QUAST y mostramos las métricas más importantes en una tabla y una gráfica de barras.

In [ ]:
report_path = pathlib.Path(f"{quast_out}/report.tsv")

df = pd.read_csv(report_path, sep="\t", header=0, index_col=0)

# Tabla completa
print("=== Reporte QUAST completo ===")
print(df.to_string())

# Métricas clave para visualización
METRICAS_CLAVE = [
    "# contigs",
    "Largest contig",
    "Total length",
    "N50",
    "L50",
    "Genome fraction (%)",
    "Mismatches per 100 kbp",
    "Indels per 100 kbp",
]

disponibles = [m for m in METRICAS_CLAVE if m in df.index]
df_key = df.loc[disponibles]

print("\n=== Métricas clave ===")
print(df_key.to_string())

In [ ]:
# Gráfico de métricas clave de QUAST
# Seleccionamos métricas numéricas de interés para visualizar
METRICAS_PLOT = {
    "N50": "N50 (pb)",
    "Genome fraction (%)": "Fracción del genoma (%)",
    "# contigs": "Número de contigs",
    "Mismatches per 100 kbp": "Mismatches / 100 kbp",
}

col = df.columns[0]  # nombre de la columna de resultados

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f'Métricas QUAST — {CASO}', fontsize=14, fontweight='bold')
axes = axes.flatten()

for ax, (metrica, etiqueta) in zip(axes, METRICAS_PLOT.items()):
    if metrica in df.index:
        valor = float(str(df.loc[metrica, col]).replace(",", ""))
        color = 'steelblue' if 'contigs' not in metrica.lower() else 'tomato'
        ax.bar([etiqueta], [valor], color=color, edgecolor='white')
        ax.set_title(etiqueta, fontsize=10)
        ax.set_ylabel("Valor")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    else:
        ax.text(0.5, 0.5, f'No disponible:\n{metrica}',
                ha='center', va='center', transform=ax.transAxes, color='gray')
        ax.set_title(etiqueta, fontsize=10)

plt.tight_layout()

quast_plot_path = f"{quast_out}/quast_metrics_{CASO}.png"
plt.savefig(quast_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Gráfico QUAST guardado en {quast_plot_path}")

## 📦 Paso 7 — Comprimir todos los resultados

In [ ]:
all_results_archive = f"{CASO}/all_results.zip"

# Comprimir toda la carpeta de resultados
!zip -r "{all_results_archive}" "{CASO}/results"

print(f"\n✅ Todos los resultados comprimidos en: {all_results_archive}")
print("Puedes descargar este archivo desde el explorador de archivos de Colab.")

## ❓ Preguntas para reflexionar

1. ¿Cuántos contigs produjo SPAdes? ¿Cómo se compara con lo esperado para una bacteria de este tamaño?
2. ¿Cuál es el N50 y el L50 del ensamblaje? ¿Qué indican?
3. ¿Qué porcentaje del genoma de referencia fue cubierto (*Genome Fraction*)?
4. ¿Cuántos mismatches e indels por 100 kbp reporta QUAST?
5. Compare la calidad Q30 antes y después del trimming con fastp. ¿Mejoró significativamente?
6. **Caso C solamente:** ¿Cómo se refleja el alto contenido GC (~72%) en los resultados de fastp y en el ensamblaje?

---
*Para repasar los conceptos de N50, L50, Genome Fraction y métricas de evaluación, consulte las secciones 6 y 7.5 del [README del Módulo 5](../README.md).*